# 08 LoRA MVP v4 Checkpoint Sweep Colab

v4 延续 v3 已验证有效的方向：99 个 audio tower attention + speech projection target、noise/reverb-only 训练、`scenario + text_length_bucket` 长短均衡采样。和 v3 的区别是：训练到 600 step，并每 160 step 保存一个中间 adapter，然后在同一固定 MVP 150 held-out test 上逐个评测 checkpoint，选择最接近或超过 10% 相对改善且 clean 无退化的 checkpoint。


In [ ]:
# 挂载 Google Drive。
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# 安装最小依赖。
# 当前训练和评测不依赖 torchao；Colab 旧版 torchao 会导致 PEFT 注入/加载 LoRA 失败。
%pip -q install --upgrade --upgrade-strategy only-if-needed qwen-asr==0.0.6 transformers==4.57.6 accelerate==1.12.0 peft==0.19.1 bitsandbytes huggingface_hub pyyaml
%pip -q install pandas==2.2.2 requests==2.32.4
%pip -q uninstall -y torchao


In [ ]:
# v4 实验参数集中在这里。
# 这轮优先比较 160/320/480/600 step checkpoint，不改 target 和数据方向。
from pathlib import Path
import json
import yaml

PROJECT_DIR = Path('/content/drive/MyDrive/qwen3-asr')
CONFIG_PATH = PROJECT_DIR / 'configs/train/qwen3_asr_lora_mvp_v4_checkpoint_sweep.yaml'
TRAIN_MANIFEST = PROJECT_DIR / 'data/jsonl/lora_mvp_train.local.jsonl'
HELD_OUT_MANIFEST = PROJECT_DIR / 'data/jsonl/baseline_mvp_150.local.jsonl'
BASE_RECHECK_METRICS = PROJECT_DIR / 'outputs/base_recheck_mvp_150/metrics.qwen3_asr_base_recheck.mvp_150.json'
HISTORICAL_BASE_METRICS = PROJECT_DIR / 'outputs/baseline_mvp_150/metrics.qwen3_asr_base.mvp_150.json'
V1_METRICS = PROJECT_DIR / 'outputs/lora_mvp_eval/metrics.qwen3_asr_lora_mvp.mvp_150.json'
V2_METRICS = PROJECT_DIR / 'outputs/lora_mvp_v2_eval/metrics.qwen3_asr_lora_mvp_v2.mvp_150.json'
V3_METRICS = PROJECT_DIR / 'outputs/lora_mvp_v3_eval/metrics.qwen3_asr_lora_mvp_v3.mvp_150.json'

config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
MODEL_ID = config.get('model', {}).get('id', 'Qwen/Qwen3-ASR-1.7B')
DTYPE = config.get('probe', {}).get('dtype', 'float16')
DEVICE_MAP = config.get('probe', {}).get('device_map', 'cuda:0')
QUANTIZATION = config.get('model', {}).get('quantization', '4bit')
LANGUAGE = 'English'
MAX_NEW_TOKENS = 128
MAX_INFERENCE_BATCH_SIZE = 1

OUTPUT_DIR = PROJECT_DIR / config.get('output', {}).get('checkpoint_dir', 'checkpoints/qwen3-asr-1.7b-lora-mvp-v4-checkpoint-sweep')
EVAL_DIR = PROJECT_DIR / config.get('output', {}).get('eval_dir', 'outputs/lora_mvp_v4_eval')
FINAL_ADAPTER_DIR = OUTPUT_DIR / 'adapter'
INCLUDE_SCENARIOS = ','.join(config.get('training', {}).get('include_scenarios', ['noise', 'reverb']))
MAX_STEPS = int(config.get('training', {}).get('max_steps', 600))
SAVE_STEPS = int(config.get('output', {}).get('save_steps', 150))
KEEP_LAST_CHECKPOINTS = int(config.get('output', {}).get('keep_last_checkpoints', 4))
LEARNING_RATE = float(config.get('training', {}).get('learning_rate', 2e-5))
EXPECTED_TARGET_COUNT = int(config.get('lora', {}).get('expected_target_count', 99))
SAMPLING_STRATEGY = config.get('training', {}).get('sampling_strategy', 'manifest_order')
SAMPLING_BUCKET_FIELDS = ','.join(config.get('training', {}).get('sampling_bucket_fields', []))
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('CONFIG_PATH =', CONFIG_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('EVAL_DIR =', EVAL_DIR)
print('INCLUDE_SCENARIOS =', INCLUDE_SCENARIOS)
print('MAX_STEPS =', MAX_STEPS, 'SAVE_STEPS =', SAVE_STEPS, 'LEARNING_RATE =', LEARNING_RATE)
print('EXPECTED_TARGET_COUNT =', EXPECTED_TARGET_COUNT)
print('SAMPLING_STRATEGY =', SAMPLING_STRATEGY, 'SAMPLING_BUCKET_FIELDS =', SAMPLING_BUCKET_FIELDS)


In [ ]:
# 检查 GPU 和必要文件。
import subprocess
import torch

print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    print('capability =', torch.cuda.get_device_capability(0))
else:
    raise RuntimeError('当前 runtime 没有 CUDA GPU，无法训练 Qwen3-ASR LoRA。')
subprocess.run(['nvidia-smi'], check=False)

required = [
    CONFIG_PATH,
    TRAIN_MANIFEST,
    HELD_OUT_MANIFEST,
    BASE_RECHECK_METRICS,
    HISTORICAL_BASE_METRICS,
    V1_METRICS,
    V2_METRICS,
    V3_METRICS,
    PROJECT_DIR / 'train/train_qwen3_asr_lora.py',
    PROJECT_DIR / 'inference/qwen3_asr_lora_infer.py',
    PROJECT_DIR / 'evaluation/eval_wer.py',
    PROJECT_DIR / 'evaluation/analyze_errors.py',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('缺少必要文件:\n' + '\n'.join(missing))


In [ ]:
# 检查训练 manifest，并确认 600 step 会均衡覆盖 noise/reverb 的 short/long。
from collections import Counter, defaultdict


def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def resolve_audio(audio):
    p = Path(audio)
    if p.is_absolute():
        return p
    return PROJECT_DIR / p


def balanced_round_robin(rows, fields):
    buckets = defaultdict(list)
    for row in rows:
        key = tuple(str(row.get(field, '')) for field in fields)
        buckets[key].append(row)
    if len(buckets) <= 1:
        return rows
    ordered = []
    for index in range(max(len(items) for items in buckets.values())):
        for key in sorted(buckets):
            items = buckets[key]
            if index < len(items):
                ordered.append(items[index])
    return ordered

train_rows = read_jsonl(TRAIN_MANIFEST)
scenario_set = {item.strip() for item in INCLUDE_SCENARIOS.split(',') if item.strip()}
bucket_fields = [item.strip() for item in SAMPLING_BUCKET_FIELDS.split(',') if item.strip()]
selected_rows = [row for row in train_rows if row.get('scenario') in scenario_set]
ordered_rows = balanced_round_robin(selected_rows, bucket_fields) if SAMPLING_STRATEGY == 'scenario_bucket_round_robin' else selected_rows
preview_rows = [ordered_rows[(step - 1) % len(ordered_rows)] for step in range(1, MAX_STEPS + 1)]

print('train rows =', len(train_rows), 'selected =', len(selected_rows), 'preview_steps =', len(preview_rows))
print('selected scenario counts =', dict(Counter(row.get('scenario', '') for row in selected_rows)))
print('selected scenario/bucket counts =', dict(Counter((row.get('scenario', ''), row.get('text_length_bucket', '')) for row in selected_rows)))
print('preview scenario/bucket counts =', dict(Counter((row.get('scenario', ''), row.get('text_length_bucket', '')) for row in preview_rows)))
print('first 12 ordered rows =', [(row.get('scenario'), row.get('text_length_bucket'), row.get('utterance_id')) for row in preview_rows[:12]])

missing = [str(resolve_audio(row['audio'])) for row in selected_rows if not resolve_audio(row['audio']).exists()]
print('missing selected audio =', len(missing))
if missing:
    print('\n'.join(missing[:20]))
    raise FileNotFoundError(f'v4 selected train rows 有缺失音频: {len(missing)}')
if not selected_rows:
    raise ValueError('scenario filter 没有选中任何训练样本')

expected_buckets = {(scenario, bucket) for scenario in scenario_set for bucket in {'short', 'long'}}
covered_buckets = {(row.get('scenario', ''), row.get('text_length_bucket', '')) for row in preview_rows}
missing_buckets = expected_buckets - covered_buckets
if missing_buckets:
    raise RuntimeError(f'{MAX_STEPS} step 预览没有覆盖所有 scenario/bucket: {sorted(missing_buckets)}')


In [ ]:
# 可选：设置 Hugging Face token。
# 如果模型下载遇到权限或限流问题，在 Colab Secrets 里设置 HF_TOKEN 后重跑本 cell。
import os

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token detected from Colab Secrets.')
else:
    print('No HF token found in Colab Secrets. Public download will be used.')


In [ ]:
# 命令执行工具：打印 stdout/stderr tail，便于快速定位失败点。
import subprocess
import sys


def run_cmd(cmd, stderr_tail=16000, stdout_tail=12000):
    print('运行命令:')
    print(' '.join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=str(PROJECT_DIR), text=True, capture_output=True)
    print('returncode =', result.returncode)
    if result.stdout:
        print('--- stdout tail ---')
        print(result.stdout[-stdout_tail:])
    if result.stderr:
        print('--- stderr tail ---')
        print(result.stderr[-stderr_tail:])
    result.check_returncode()
    return result


In [ ]:
# Preflight：确认 v4 checkpoint-sweep target 数为 99。
preflight_cmd = [
    sys.executable,
    'train/train_qwen3_asr_lora.py',
    '--config', str(CONFIG_PATH),
    '--manifest', str(TRAIN_MANIFEST),
    '--audio-root', str(PROJECT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--model-id', MODEL_ID,
    '--dtype', DTYPE,
    '--device-map', DEVICE_MAP,
    '--quantization', QUANTIZATION,
    '--language', LANGUAGE,
    '--include-scenarios', INCLUDE_SCENARIOS,
    '--sampling-strategy', SAMPLING_STRATEGY,
    '--sampling-bucket-fields', SAMPLING_BUCKET_FIELDS,
    '--learning-rate', str(LEARNING_RATE),
    '--max-steps', str(MAX_STEPS),
    '--save-steps', str(SAVE_STEPS),
    '--keep-last-checkpoints', str(KEEP_LAST_CHECKPOINTS),
    '--preflight-only',
]
run_cmd(preflight_cmd)

summary = json.loads((OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2)[:5000])
assert summary.get('status') == 'preflight_ok', summary
assert summary.get('count') == EXPECTED_TARGET_COUNT, summary.get('count')
print('preflight 验收通过。')


In [ ]:
# v4 checkpoint-sweep 训练：默认 600 step，每 160 step 保存一次中间 adapter。
train_cmd = [
    sys.executable,
    'train/train_qwen3_asr_lora.py',
    '--config', str(CONFIG_PATH),
    '--manifest', str(TRAIN_MANIFEST),
    '--audio-root', str(PROJECT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--model-id', MODEL_ID,
    '--dtype', DTYPE,
    '--device-map', DEVICE_MAP,
    '--quantization', QUANTIZATION,
    '--language', LANGUAGE,
    '--include-scenarios', INCLUDE_SCENARIOS,
    '--sampling-strategy', SAMPLING_STRATEGY,
    '--sampling-bucket-fields', SAMPLING_BUCKET_FIELDS,
    '--learning-rate', str(LEARNING_RATE),
    '--max-steps', str(MAX_STEPS),
    '--save-steps', str(SAVE_STEPS),
    '--keep-last-checkpoints', str(KEEP_LAST_CHECKPOINTS),
]
run_cmd(train_cmd, stderr_tail=18000, stdout_tail=14000)

summary = json.loads((OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
loss_lines = [line for line in (OUTPUT_DIR / 'loss_log.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
print(json.dumps(summary, ensure_ascii=False, indent=2)[:5000])
print('loss lines =', len(loss_lines))
print('last losses:')
print('\n'.join(loss_lines[-10:]))
loss_rows = [json.loads(line) for line in loss_lines]
loss_bucket_counts = Counter((row.get('scenario', ''), row.get('text_length_bucket', '')) for row in loss_rows)
print('loss scenario/bucket counts =', dict(loss_bucket_counts))
expected_buckets = {(scenario, bucket) for scenario in scenario_set for bucket in {'short', 'long'}}
missing_loss_buckets = expected_buckets - set(loss_bucket_counts)
if missing_loss_buckets:
    raise RuntimeError(f'loss log 没有覆盖所有 scenario/bucket: {sorted(missing_loss_buckets)}')
assert summary.get('status') == 'trained', summary
assert summary.get('steps') == MAX_STEPS, summary.get('steps')
assert FINAL_ADAPTER_DIR.exists(), f'缺少 final adapter dir: {FINAL_ADAPTER_DIR}'
assert len(loss_lines) == MAX_STEPS, len(loss_lines)

checkpoint_adapters = []
for item in summary.get('saved_checkpoints', []):
    step = int(item['step'])
    adapter = Path(item['adapter_dir'])
    if not adapter.is_absolute():
        adapter = PROJECT_DIR / adapter
    checkpoint_adapters.append((step, adapter))
checkpoint_adapters.append((MAX_STEPS, FINAL_ADAPTER_DIR))
print('checkpoint adapters =')
for step, adapter in checkpoint_adapters:
    print(step, adapter, adapter.exists())
    assert adapter.exists(), adapter


In [ ]:
# 逐个 checkpoint 跑 held-out MVP 150 推理、WER/CER 和错误分析。
# 注意：每个 checkpoint 会重新加载一次 base + adapter，耗时较长，但比较最公平。
checkpoint_results = []

for step, adapter_dir in checkpoint_adapters:
    run_name = f'step_{step:04d}'
    run_eval_dir = EVAL_DIR / run_name
    run_eval_dir.mkdir(parents=True, exist_ok=True)
    pred_jsonl = run_eval_dir / f'predictions.qwen3_asr_lora_mvp_v4_{run_name}.mvp_150.jsonl'
    scored_jsonl = run_eval_dir / f'predictions.qwen3_asr_lora_mvp_v4_{run_name}.mvp_150.scored.jsonl'
    metrics_json = run_eval_dir / f'metrics.qwen3_asr_lora_mvp_v4_{run_name}.mvp_150.json'
    scenario_csv = run_eval_dir / f'metrics_by_scenario.qwen3_asr_lora_mvp_v4_{run_name}.mvp_150.csv'
    error_dir = run_eval_dir / 'error_analysis'

    infer_cmd = [
        sys.executable,
        'inference/qwen3_asr_lora_infer.py',
        '--manifest', str(HELD_OUT_MANIFEST),
        '--output-jsonl', str(pred_jsonl),
        '--adapter-dir', str(adapter_dir),
        '--audio-root', str(PROJECT_DIR),
        '--model-id', MODEL_ID,
        '--dtype', DTYPE,
        '--device-map', DEVICE_MAP,
        '--quantization', QUANTIZATION,
        '--max-inference-batch-size', str(MAX_INFERENCE_BATCH_SIZE),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--language', LANGUAGE,
    ]
    run_cmd(infer_cmd, stderr_tail=18000, stdout_tail=14000)

    eval_cmd = [
        sys.executable,
        'evaluation/eval_wer.py',
        '--predictions-jsonl', str(pred_jsonl),
        '--scored-jsonl', str(scored_jsonl),
        '--metrics-json', str(metrics_json),
        '--metrics-by-scenario-csv', str(scenario_csv),
    ]
    run_cmd(eval_cmd)

    analysis_cmd = [
        sys.executable,
        'evaluation/analyze_errors.py',
        '--scored-jsonl', str(scored_jsonl),
        '--output-dir', str(error_dir),
    ]
    run_cmd(analysis_cmd)

    checkpoint_results.append({
        'step': step,
        'adapter_dir': str(adapter_dir),
        'eval_dir': str(run_eval_dir),
        'metrics_json': str(metrics_json),
        'scored_jsonl': str(scored_jsonl),
    })

print(json.dumps(checkpoint_results, ensure_ascii=False, indent=2))


In [ ]:
# 汇总 base/v1/v2/v3/v4 checkpoint-sweep 对比，选择候选最佳 checkpoint。
import pandas as pd


def load_metrics(path):
    data = json.loads(Path(path).read_text(encoding='utf-8'))
    out = {row['group']: row for row in data.get('by_scenario', [])}
    out['ALL'] = data.get('overall', {})
    return out


def wer(metrics, scenario):
    return float(metrics.get(scenario, {}).get('error_rate', 0.0) or 0.0)


def edits(metrics, scenario):
    return int(metrics.get(scenario, {}).get('num_edits', 0) or 0)


def refs(metrics, scenario):
    return int(metrics.get(scenario, {}).get('ref_len', 0) or 0)

base = load_metrics(BASE_RECHECK_METRICS)
historical = load_metrics(HISTORICAL_BASE_METRICS)
v1 = load_metrics(V1_METRICS)
v2 = load_metrics(V2_METRICS)
v3 = load_metrics(V3_METRICS)
scenarios = ['ALL', 'clean', 'noise', 'reverb', 'dropout', 'far_field']
rows = []

for result in checkpoint_results:
    m = load_metrics(result['metrics_json'])
    target_base_edits = edits(base, 'noise') + edits(base, 'reverb')
    target_base_refs = refs(base, 'noise') + refs(base, 'reverb')
    target_edits = edits(m, 'noise') + edits(m, 'reverb')
    target_refs = refs(m, 'noise') + refs(m, 'reverb')
    target_base_wer = target_base_edits / target_base_refs
    target_wer = target_edits / target_refs
    rows.append({
        'step': result['step'],
        'overall': wer(m, 'ALL'),
        'clean': wer(m, 'clean'),
        'noise': wer(m, 'noise'),
        'reverb': wer(m, 'reverb'),
        'dropout': wer(m, 'dropout'),
        'far_field': wer(m, 'far_field'),
        'noise_rel_vs_base': (wer(m, 'noise') - wer(base, 'noise')) / wer(base, 'noise'),
        'reverb_rel_vs_base': (wer(m, 'reverb') - wer(base, 'reverb')) / wer(base, 'reverb'),
        'target_rel_vs_base': (target_wer - target_base_wer) / target_base_wer,
        'clean_rel_vs_base': (wer(m, 'clean') - wer(base, 'clean')) / wer(base, 'clean') if wer(base, 'clean') else None,
        'metrics_json': result['metrics_json'],
        'adapter_dir': result['adapter_dir'],
    })

summary_df = pd.DataFrame(rows).sort_values(['target_rel_vs_base', 'noise_rel_vs_base', 'reverb_rel_vs_base'])
display(summary_df)

print('Reference WERs:')
print('historical base overall/noise/reverb =', wer(historical, 'ALL'), wer(historical, 'noise'), wer(historical, 'reverb'))
print('4bit base overall/noise/reverb =', wer(base, 'ALL'), wer(base, 'noise'), wer(base, 'reverb'))
print('v1 overall/noise/reverb =', wer(v1, 'ALL'), wer(v1, 'noise'), wer(v1, 'reverb'))
print('v2 overall/noise/reverb =', wer(v2, 'ALL'), wer(v2, 'noise'), wer(v2, 'reverb'))
print('v3 overall/noise/reverb =', wer(v3, 'ALL'), wer(v3, 'noise'), wer(v3, 'reverb'))

eligible = summary_df[
    ((summary_df['noise_rel_vs_base'] <= -0.10) | (summary_df['reverb_rel_vs_base'] <= -0.10) | (summary_df['target_rel_vs_base'] <= -0.10))
    & ((summary_df['clean_rel_vs_base'].isna()) | (summary_df['clean_rel_vs_base'] <= 0.05))
]
if len(eligible):
    best = eligible.iloc[0]
    print('v4 找到候选 checkpoint：')
    print(best.to_dict())
else:
    best = summary_df.iloc[0]
    print('v4 还未达到 10% 门槛；当前相对最优 checkpoint：')
    print(best.to_dict())


In [ ]:
# 列出 v4 产物，后续可用 00_github_commit_push_colab.ipynb 提交受控输出。
for root in [OUTPUT_DIR, EVAL_DIR]:
    print('\n===', root, '===')
    for path in sorted(root.rglob('*')):
        if path.is_file():
            print(path.relative_to(PROJECT_DIR), path.stat().st_size)
